<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Top_Reversal/Top_Reversal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Stock Reversal Scanner

This scanner identifies potential top reversal candidates based on three criteria:

1.  **90-Day High in Last 3 Days**: The stock's price has reached its 90-day high within the last three trading days, indicating recent strong upward momentum.
2.  **Significant Run-Up**: The stock has experienced a substantial percentage increase over the last 5 trading days, suggesting it might be overextended and vulnerable to mean reversion.
3.  **Close Below Previous Day's Low**: The stock closed below the previous day's low, indicating a potential shift in momentum from bullish to bearish.

The script will iterate through the tickers in your `watchlist_df`, download historical data, and apply these rules. Stocks that meet all three criteria will be listed as reversal candidates.

In [6]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

In [7]:
# Clear all DataFrames from memory
import gc

# Get a list of all variables in the global namespace
all_vars = list(globals().keys())

# Identify and delete pandas DataFrames
for var_name in all_vars:
    if isinstance(globals()[var_name], pd.DataFrame):
        del globals()[var_name]
        print(f"Deleted DataFrame: {var_name}")

# Run garbage collector to free up memory
gc.collect()

print("All DataFrames cleared from memory.")

Deleted DataFrame: option_volume_df
Deleted DataFrame: watchlist_df
All DataFrames cleared from memory.


In [8]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'

OptionVolume200_id = '1gcwD510l4GFGNcKsbExR3GvKnDZwCHy4'
OptionVolume200 = f'https://drive.google.com/uc?export=download&id={OptionVolume200_id}'

In [9]:
# Load the OptionVolume data to create the watchlist
option_volume_df = pd.read_csv(OptionVolume)
watchlist_df = pd.DataFrame(option_volume_df['Symbol'].unique(), columns=['Ticker'])

print(f"Watchlist created with {len(watchlist_df)} unique tickers.")
display(watchlist_df.head())

Watchlist created with 100 unique tickers.


,Ticker
0,SPY
1,TSLA
2,QQQ
3,NVDA
4,MSFT


In [10]:
# --- Configuration --- #
start_scan_date = datetime.strptime('2026-01-01', '%Y-%m-%d')
end_scan_date = datetime.strptime('2026-06-18', '%Y-%m-%d')
lookback_days_for_high = 90  # For 90-day high

all_tickers_data = {}
error_count = 0
skipped_count = 0

print(f"Scanning and downloading historical data from {start_scan_date.strftime('%Y-%m-%d')} to {end_scan_date.strftime('%Y-%m-%d')}...")

overall_start_download_date = start_scan_date - timedelta(days=lookback_days_for_high + 10)
overall_end_download_date = end_scan_date + timedelta(days=1)

for ticker in watchlist_df['Ticker'].unique():
    try:
        # Fetching data with multi_level_index disabled to ensure stable columns
        data = yf.download(
            ticker,
            start=overall_start_download_date,
            end=overall_end_download_date,
            progress=False,
            auto_adjust=True,
            multi_level_index=False
        )

        if data.empty:
            print(f"  [Skipped] {ticker}: No data found for the specified period.")
            skipped_count += 1
            continue

        data = data.sort_index()

        # Clean explicit column layout verification
        rename_map = {}
        if 'Adj Close' in data.columns and 'Close' not in data.columns:
            rename_map['Adj Close'] = 'Close'

        if rename_map:
            data = data.rename(columns=rename_map)

        expected_ohclv_cols = ['Open', 'High', 'Low', 'Close', 'Volume']

        if not all(col in data.columns for col in expected_ohclv_cols):
            print(f"  [Error] {ticker}: Missing essential OHLCV columns. Found: {data.columns.tolist()}")
            error_count += 1
            continue

        all_tickers_data[ticker] = data[expected_ohclv_cols]

    except Exception as e:
        print(f"  [Exception] Error processing {ticker}: {e}")
        error_count += 1

# Final Summary Terminal Output
print("\n" + "="*40)
print(f"Download complete.")
print(f"  Successfully processed: {len(all_tickers_data)} tickers")
if skipped_count > 0:
    print(f"  Skipped (Empty data): {skipped_count}")
if error_count > 0:
    print(f"  Errors encountered:   {error_count}")
print("="*40 + "\n")

Scanning and downloading historical data from 2026-01-01 to 2026-06-18...

Download complete.
  Successfully processed: 100 tickers



In [11]:

sma_lookback = 10            # Moving average lookback period
pct_above_sma_threshold = 0.05 # Configurable threshold (0.05 = 5% above SMA)

reversal_candidates = []

for ticker, data in all_tickers_data.items():
    try:
        df = data.sort_index().copy()
        if len(df) < max(lookback_days_for_high, sma_lookback + 2):
            continue

        # 1. Generate core, independent technical metrics
        df['90_Day_High'] = df['High'].rolling(window=lookback_days_for_high).max()
        df['Recent_High'] = df['High'].rolling(window=3).max()
        df['SMA_10'] = df['Close'].rolling(window=sma_lookback).mean()

        # 2. Build explicit, un-shifted daily criteria components
        df['C1_High_Met'] = df['Recent_High'] >= df['90_Day_High']
        df['C2_Distance'] = (df['Close'] - df['SMA_10']) / df['SMA_10']

        # 3. Step forward to "Today" and look backward cleanly using explicit shifts
        df['Criterion1_Met'] = df['C1_High_Met'] # True if 90-day high occurred within last 3 days
        df['Criterion2_Met'] = df['C2_Distance'].shift(1) >= pct_above_sma_threshold # Yesterday was extended
        df['Criterion3_Met'] = df['Close'] < df['Low'].shift(1) # Today closed below yesterday's low

        # Combine conditions
        df['Is_Candidate'] = df['Criterion1_Met'] & df['Criterion2_Met'] & df['Criterion3_Met']

        # Slice for the requested scan range
        valid_signals = df[df['Is_Candidate'] & (df.index >= start_scan_date) & (df.index <= end_scan_date)]

        for match_date, row in valid_signals.iterrows():
            loc = df.index.get_loc(match_date)

            # Pull values directly using exact positional offsets relative to the match day
            yesterday_pct_above_sma = df['C2_Distance'].iloc[loc - 1] * 100
            yesterday_low = df['Low'].iloc[loc - 1]

            reversal_candidates.append({
                'Scan Date': match_date.strftime('%Y-%m-%d'),
                'Ticker': ticker,
                'Latest Close': round(float(row['Close']), 2),
                'Previous Day Low': round(float(yesterday_low), 2),
                '90-Day High': round(float(row['90_Day_High']), 2),
                'Pct Above SMA': round(float(yesterday_pct_above_sma), 2)
            })

    except Exception:
        continue

# Final Output Configuration & Display
if reversal_candidates:
    reversal_df = pd.DataFrame(reversal_candidates)

    # Sort by Scan Date ascending (Oldest to Newest)
    reversal_df['Scan Date'] = pd.to_datetime(reversal_df['Scan Date'])
    reversal_df = reversal_df.sort_values(by='Scan Date', ascending=True)
    reversal_df['Scan Date'] = reversal_df['Scan Date'].dt.strftime('%Y-%m-%d')

    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(reversal_df)
else:
    print("No candidates met all three criteria in the specified range.")

,Scan Date,Ticker,Latest Close,Previous Day Low,90-Day High,Pct Above SMA
105,2026-01-30,STX,406.98,433.24,457.04,23.06
1,2026-01-30,MU,414.71,417.52,455.31,11.14
61,2026-01-30,ASTS,111.21,113.35,129.89,8.78
22,2026-01-30,SLV,75.44,96.74,109.83,14.15
76,2026-01-30,WDC,250.05,268.15,285.21,14.11
164,2026-01-30,FCX,60.10,62.97,69.29,6.53
24,2026-01-30,GLD,444.95,468.51,509.70,8.60
134,2026-01-30,LRCX,233.02,236.37,251.39,8.90
51,2026-01-30,SOXL,61.79,62.65,71.98,11.54
37,2026-02-02,USO,75.33,77.65,80.38,6.45


In [12]:
# Create a copy to prevent SettingWithCopyWarning
performance_df = reversal_df.copy()

# Ensure Scan Date is in datetime format for accurate matching
performance_df['Scan Date'] = pd.to_datetime(performance_df['Scan Date'])

# Initialize empty lists to store the calculated forward returns
returns_data = {f'Day {i} Return %': [] for i in range(1, 6)}

for _, row in performance_df.iterrows():
    ticker = row['Ticker']
    scan_date = row['Scan Date']

    # Check if we have historical data for this ticker
    if ticker in all_tickers_data:
        df_ticker = all_tickers_data[ticker].sort_index()

        try:
            # Find the positional integer index of the trigger day
            trigger_loc = df_ticker.index.get_loc(scan_date)
            trigger_close = float(df_ticker['Close'].iloc[trigger_loc])

            # Calculate forward returns for days 1 through 5
            for day in range(1, 6):
                target_loc = trigger_loc + day

                # Check if the target look-ahead row exists in our data
                if target_loc < len(df_ticker):
                    forward_close = float(df_ticker['Close'].iloc[target_loc])
                    # Formula: (Forward Close - Trigger Close) / Trigger Close
                    pct_return = ((forward_close - trigger_close) / trigger_close) * 100
                    returns_data[f'Day {day} Return %'].append(round(pct_return, 2))
                else:
                    # Append NaN if there aren't enough future trading days yet
                    returns_data[f'Day {day} Return %'].append(np.nan)

        except KeyError:
            # Handle cases where the scan date doesn't match the index cleanly
            for day in range(1, 6):
                returns_data[f'Day {day} Return %'].append(np.nan)
    else:
        # Handle cases where the ticker data is missing
        for day in range(1, 6):
            returns_data[f'Day {day} Return %'].append(np.nan)

# Merge the calculated performance metrics back into the main DataFrame
for col_name, returns_list in returns_data.items():
    performance_df[col_name] = returns_list

# Revert Scan Date back to clean string format for presentation
performance_df['Scan Date'] = performance_df['Scan Date'].dt.strftime('%Y-%m-%d')

# Display the performance breakdown
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(performance_df)

,Scan Date,Ticker,Latest Close,Previous Day Low,90-Day High,Pct Above SMA,Day 1 Return %,Day 2 Return %,Day 3 Return %,Day 4 Return %,Day 5 Return %
105,2026-01-30,STX,406.98,433.24,457.04,23.06,6.20,9.02,2.68,-0.55,5.31
1,2026-01-30,MU,414.71,417.52,455.31,11.14,5.52,1.10,-8.55,-7.71,-4.87
61,2026-01-30,ASTS,111.21,113.35,129.89,8.78,-5.99,4.09,-6.93,-16.13,-8.47
22,2026-01-30,SLV,75.44,96.74,109.83,14.15,-3.98,2.01,4.96,-11.60,-6.96
76,2026-01-30,WDC,250.05,268.15,285.21,14.11,7.99,15.99,7.66,3.98,12.93
164,2026-01-30,FCX,60.10,62.97,69.29,6.53,0.88,7.37,2.71,-1.68,0.73
24,2026-01-30,GLD,444.95,468.51,509.70,8.60,-4.00,2.10,2.03,-0.69,2.36
134,2026-01-30,LRCX,233.02,236.37,251.39,8.90,1.73,-1.44,-10.14,-8.63,-1.05
51,2026-01-30,SOXL,61.79,62.65,71.98,11.54,5.52,-0.95,-13.92,-13.82,-0.06
37,2026-02-02,USO,75.33,77.65,80.38,6.45,2.84,3.39,1.81,2.20,3.57


In [13]:
import pandas as pd

# Define the return columns we want to analyze
return_cols = [f'Day {i} Return %' for i in range(1, 6)]

summary_metrics = []

for col in return_cols:
    # Drop NaN values to ensure accurate calculations for recent triggers
    valid_returns = performance_df[col].dropna()

    if len(valid_returns) > 0:
        avg_return = valid_returns.mean()

        # Count sessions where the asset dropped (Negative Return = Strategy Success)
        negative_trades = valid_returns[valid_returns < 0]
        pct_negative = (len(negative_trades) / len(valid_returns)) * 100

        summary_metrics.append({
            'Time Period': col.replace(' Return %', ''),
            'Sample Size': len(valid_returns),
            'Average Return %': round(float(avg_return), 2),
            'Win Rate (% Negative)': round(float(pct_negative), 2)
        })
    else:
        summary_metrics.append({
            'Time Period': col.replace(' Return %', ''),
            'Sample Size': 0,
            'Average Return %': 0.0,
            'Win Rate (% Negative)': 0.0
        })

# Convert to DataFrame for a clean, structural display
summary_df = pd.DataFrame(summary_metrics)

print("=" * 50)
print("       REVERSAL STRATEGY PERFORMANCE SUMMARY       ")
print("=" * 50)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(summary_df)


       REVERSAL STRATEGY PERFORMANCE SUMMARY       


,Time Period,Sample Size,Average Return %,Win Rate (% Negative)
0,Day 1,165,0.46,43.03
1,Day 2,164,1.26,45.12
2,Day 3,156,0.24,49.36
3,Day 4,155,1.17,47.10
4,Day 5,155,2.15,48.39
